In [ ]:
from pathlib import Path
ROOT = Path('..').resolve()
DATA_DIR = ROOT / 'data'
IMG_DIR = DATA_DIR / 'images'
CKPT_DIR = ROOT / 'checkpoints'

In [ ]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [ ]:
import pandas as pd
import torch

data = pd.read_csv(f'{DATA_DIR}/dataset_final.csv')
print(data.head())

In [ ]:
import matplotlib.pyplot as plt
emotion_distribution = data['emotion'].value_counts()
plt.figure(figsize=(8, 5))
emotion_distribution.plot(kind='bar')
plt.title('Emotion Distribution')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print(emotion_distribution.describe())


utterance_lengths = data['utterance'].apply(lambda x: len(str(x).split()))
plt.figure(figsize=(8, 5))
plt.hist(utterance_lengths, bins=20)
plt.title('Utterance Length Distribution')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()
print(utterance_lengths.describe())

image_coverage = data['filename'].nunique()
print(f'Unique images: {image_coverage}')
actual_images_count = len(list(IMG_DIR.glob('*.jpg')))
print(f'Actual images in directory: {actual_images_count}')

invalid_images = set(data['filename']) - set(img.name for img in IMG_DIR.glob('*.jpg'))
print(f'Invalid image references: {len(invalid_images)}')

annotations_per_image = data.groupby('filename').size()
print(annotations_per_image.describe())

print(data.shape)

In [ ]:
from collections import Counter
import string
MIN_FREQUENCY = 2

tokenized_utterances = data['utterance'].apply(lambda x: [word.strip(string.punctuation) for word in str(x).lower().split()])
word_counts = Counter(word for utterance in tokenized_utterances for word in utterance)
vocab = {word for word in word_counts if word_counts[word] >= MIN_FREQUENCY}
vocab.update({'<PAD>', '<UNK>', '<SOS>', '<EOS>'})

word2idx = {word: idx for idx, word in enumerate(sorted(vocab))}
idx2word = {idx: word for word, idx in word2idx.items()}

print(f'Vocabulary size: {len(vocab)}')
print(f'Index for special tokens: <PAD>={word2idx["<PAD>"]}, <UNK>={word2idx["<UNK>"]}, <SOS>={word2idx["<SOS>"]}, <EOS>={word2idx["<EOS>"]}')

sample_utterance = data['utterance'].iloc[0].lower()
tokenized = [word.strip(string.punctuation) for word in sample_utterance.split()]
indices = [word2idx.get(word, word2idx['<UNK>']) for word in tokenized]
print(f'Original: {sample_utterance}')
print(f'Tokenized: {tokenized}')
print(f'Indices: {indices}')
reconstructed = ' '.join(idx2word[idx] for idx in indices)
print(f'Reconstructed: {reconstructed}')


In [ ]:
from torchvision import transforms

image_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225])
    ])

In [ ]:
from PIL import Image

class ArtemisDataset(torch.utils.data.Dataset):
    def __init__(self, data, word2idx, img_dir, image_transform, max_seq_length=40):
        self.data = data
        self.word2idx = word2idx
        self.img_dir = img_dir
        self.image_transform = image_transform
        self.max_seq_length = max_seq_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, key):
        row = self.data.iloc[key]
        img_path = f'{self.img_dir}/{row["filename"]}'
        img = Image.open(img_path).convert('RGB')
        img_tensor = self.image_transform(img)
        
        utterance = [word.strip(string.punctuation) for word in str(row['utterance']).lower().split()]
        indices = [self.word2idx.get(word, self.word2idx['<UNK>']) for word in utterance]
        indices = [self.word2idx['<SOS>']] + indices + [self.word2idx['<EOS>']]
        if len(indices) < self.max_seq_length:
            indices += [self.word2idx['<PAD>']] * (self.max_seq_length - len(indices))
        else:
            indices = indices[:self.max_seq_length-1] + [self.word2idx['<EOS>']]
        
        return img_tensor, torch.tensor(indices)
    

In [ ]:
dataset = ArtemisDataset(data, word2idx, IMG_DIR, image_transform)
print(f'Dataset size: {len(dataset)}')
sample_img, sample_indices = dataset[0]
print(f'Sample image tensor shape: {sample_img.shape}')
print(f'Sample indices: {sample_indices}')
print(f'Sample reconstructed: {" ".join(idx2word[idx.item()] for idx in sample_indices if idx.item() in idx2word)}')

In [ ]:
from sklearn.model_selection import train_test_split
train_data, test_data = train_test_split(data, test_size=0.1, random_state=42)
train_data, val_data = train_test_split(train_data, test_size=0.111, random_state=42)

train_dataset = ArtemisDataset(train_data, word2idx, IMG_DIR, image_transform)
val_dataset = ArtemisDataset(val_data, word2idx, IMG_DIR, image_transform)
test_dataset = ArtemisDataset(test_data, word2idx, IMG_DIR, image_transform)

In [ ]:
BATCH_SIZE = 32
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [ ]:
test_batch = next(iter(train_loader))
test_images, test_sequences = test_batch
print(test_images.shape)
print(test_sequences.shape)

In [ ]:
class CNNImageEncoder(torch.nn.Module):
    def __init__(self, output_dim=512):
        super(CNNImageEncoder, self).__init__()
        self.cnn = torch.nn.Sequential(
            torch.nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            torch.nn.BatchNorm2d(32),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2),
            torch.nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            torch.nn.BatchNorm2d(64),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2),
            torch.nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            torch.nn.BatchNorm2d(128),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2),
            torch.nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            torch.nn.BatchNorm2d(256),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.fc = torch.nn.Linear(256, output_dim)
    
    def forward(self, x):
        x = self.cnn(x)
        x = x.flatten(2).transpose(1, 2)
        x = self.fc(x)
        return x

In [ ]:
x = torch.randn(2, 3, 224, 224)
encoder = CNNImageEncoder()
output = encoder(x)
print(output.shape)

In [ ]:
class SinusoidalPositionalEncoding(torch.nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(SinusoidalPositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [ ]:
pe = SinusoidalPositionalEncoding(d_model=512, max_len=40)
x = torch.randn(2, 40, 512)
print(pe(x).shape)

In [ ]:
def attention(query, key, value, mask=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
    
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    attention_weights = torch.nn.functional.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, value)
    
    return output, attention_weights

In [ ]:
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        
        self.linear_q = torch.nn.Linear(d_model, d_model)
        self.linear_k = torch.nn.Linear(d_model, d_model)
        self.linear_v = torch.nn.Linear(d_model, d_model)
        self.linear_out = torch.nn.Linear(d_model, d_model)
    
    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        
        query = self.linear_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        key = self.linear_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        value = self.linear_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        output, attention_weights = attention(query, key, value, mask)
        
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads * self.d_k)
        
        return self.linear_out(output), attention_weights

In [ ]:
class FeedForwardNetwork(torch.nn.Module):
    def __init__(self, d_model, d_feedforward, dropout=0.1):
        super(FeedForwardNetwork, self).__init__()
        self.linear1 = torch.nn.Linear(d_model, d_feedforward)
        self.dropout = torch.nn.Dropout(dropout)
        self.linear2 = torch.nn.Linear(d_feedforward, d_model)
    
    def forward(self, x):
        x = torch.nn.functional.relu(self.linear1(x))
        x = self.dropout(x)
        return self.linear2(x)

In [ ]:
class DecoderLayer(torch.nn.Module):
    def __init__(self, d_model, num_heads, d_feedforward, dropout=0.1):
        super(DecoderLayer, self).__init__()
        self.self_attention = MultiHeadAttention(d_model, num_heads)
        self.cross_attention = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForwardNetwork(d_model, d_feedforward, dropout)
        
        self.norm1 = torch.nn.LayerNorm(d_model)
        self.norm2 = torch.nn.LayerNorm(d_model)
        self.norm3 = torch.nn.LayerNorm(d_model)
        
        self.dropout1 = torch.nn.Dropout(dropout)
        self.dropout2 = torch.nn.Dropout(dropout)
        self.dropout3 = torch.nn.Dropout(dropout)
    
    def forward(self, x, memory, memory_mask=None):
        T = x.size(1)
        causal_mask = torch.tril(torch.ones((T, T), device=x.device)).unsqueeze(0).unsqueeze(0)
       
        # Self-attention
        attn_output1, _ = self.self_attention(x, x, x, causal_mask)
        x = x + self.dropout1(attn_output1)
        x = self.norm1(x)
        
        # Cross-attention
        attn_output2, _ = self.cross_attention(x, memory, memory, memory_mask)
        x = x + self.dropout2(attn_output2)
        x = self.norm2(x)
        
        # Feed-forward
        ff_output = self.feed_forward(x)
        x = x + self.dropout3(ff_output)
        x = self.norm3(x)
        
        return x

In [ ]:
class TransformerDecoder(torch.nn.Module):
    def __init__(self, vocab_size, d_model=512, num_heads=8, d_feedforward=2048, num_layers=4, dropout=0.1):
        super(TransformerDecoder, self).__init__()
        self.embedding = torch.nn.Embedding(vocab_size, d_model)
        self.positional_encoding = SinusoidalPositionalEncoding(d_model)
        self.layers = torch.nn.ModuleList([DecoderLayer(d_model, num_heads, d_feedforward, dropout) for _ in range(num_layers)])
        self.fc_out = torch.nn.Linear(d_model, vocab_size)
    
    def forward(self, tgt, memory, memory_mask=None):
        x = self.embedding(tgt)
        x = self.positional_encoding(x)
        
        for layer in self.layers:
            x = layer(x, memory, memory_mask)
        
        output = self.fc_out(x)
        return output

In [ ]:
mha = MultiHeadAttention(d_model=512, num_heads=8)
q = torch.randn(2, 40, 512)
mem = torch.randn(2, 196, 512)
out, _ = mha(q, mem, mem)
print(out.shape)

In [ ]:
decoder = TransformerDecoder(vocab_size=len(vocab),num_layers=4)
tgt = torch.randint(0, 10749, (2,40))
mem = torch.randn(2, 196, 512)
print(decoder(tgt, mem).shape)

In [ ]:
class MUSE(torch.nn.Module):
    def __init__(self, vocab_size, d_model=512, num_heads=8, d_feedforward=2048, num_layers=4, dropout=0.1):
        super(MUSE, self).__init__()
        self.encoder = CNNImageEncoder(output_dim=d_model)
        self.decoder = TransformerDecoder(vocab_size, d_model, num_heads, d_feedforward, num_layers, dropout)
    
    def forward(self, images, tgt, memory_mask=None):
        memory = self.encoder(images)
        output = self.decoder(tgt, memory, memory_mask)
        return output

In [ ]:
VOCAB_SIZE = len(vocab)
model = MUSE(vocab_size=VOCAB_SIZE)

images = torch.randn(2, 3, 224, 224)
tgt = torch.randint(0, VOCAB_SIZE, (2, 40))
output = model(images, tgt)
print(output.shape)

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss(ignore_index=word2idx['<PAD>'])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
device = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

In [ ]:
from tqdm import tqdm

def train_muse(model, train_loader, val_loader, loss_fn, optimizer, device, num_epochs=10):
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        train_bar = tqdm(train_loader, desc=f'[Train] Epoch {epoch+1}/{num_epochs}', leave=False)
        for images, tgt in train_bar:
            images, tgt = images.to(device), tgt.to(device)
            optimizer.zero_grad()
            output = model(images, tgt[:, :-1])
            train_loss = loss_fn(output.view(-1, output.size(-1)), tgt[:, 1:].reshape(-1))
            train_loss.backward()
            optimizer.step()
            total_loss += train_loss.item()
            train_bar.set_postfix({'Train Loss': train_loss.item()})
        
        avg_train_loss = total_loss / len(train_loader)
        
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            val_bar = tqdm(val_loader, desc=f'[Val] Epoch {epoch+1}/{num_epochs}', leave=False)
            for images, tgt in val_bar:
                images, tgt = images.to(device), tgt.to(device)
                output = model(images, tgt[:, :-1])
                val_loss = loss_fn(output.view(-1, output.size(-1)), tgt[:, 1:].reshape(-1))
                total_val_loss += val_loss.item()
                val_bar.set_postfix({'Val Loss': val_loss.item()})
        avg_val_loss = total_val_loss / len(val_loader)

        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': avg_train_loss,
            'val_loss': avg_val_loss,
        }, f'{CKPT_DIR}/checkpoint_epoch_{epoch}.pt')
        
        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}')

In [ ]:
train_muse(model, train_loader, val_loader, loss_fn, optimizer, device, num_epochs=10)

In [ ]:
checkpoint = torch.load(f'{CKPT_DIR}/checkpoint_epoch_9.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()